In [10]:
import pandas as pd
import os

In [3]:
df=pd.read_csv(r"C:\Uni\4th Year\GP\ECHO\data\chatbot\outputs\evaluation_data\synthetic_dataset.csv")

In [9]:
# Shuffles the whole dataset, then picks one per entity
shrunk_df = df.sample(frac=1, random_state=42).drop_duplicates(subset=['entity_name'])
shrunk_df

,input,expected_output,context,entity_name,entity_type
79,"Hathor, how did your divine influence extend s...","I am Hathor, Lady of Turquoise, Lady of the We...",. Associated with Hathor’s cult was a group of...,Hathor (Goddess),pharaoh
316,"Colossoi of Memnon, how did the Roman engineer...",When the earth heaved in 27 BCE and shattered ...,. The side panels depict the Nile god Hapi. Th...,Colossoi of Memnon,landmark
485,"Temple of Wadi es-Sebua—Temple of Ramesses II,...",I rise in the sun-blasted silence 150 metres n...,. It was built about 150 metres northeast of t...,Temple of Wadi es-Sebua - Temple of Ramesses II,landmark
396,"Sphinx, why do scholars still argue about your...",I feel their restless eyes upon my weathered b...,"Even so, the Sphinx defies such an easy and co...",Sphinx,landmark
167,"Niuserre, mighty king, can you describe for us...","I, Niuserre, ascended only after the hawk had ...",. Niuserre most likely succeeded his older bro...,Niuserre,pharaoh
...,...,...,...,...,...
186,"Mighty Psusennes I, third sovereign of the Twe...","I, Horus-strong Akheperre-setepenamun, beheld ...","Psusennes I, also known as ’Akheperre’setepena...",Psusennes I,pharaoh
51,"Beloved Queen Ankhesenamun, when you gaze upon...",I cradle the memory of those little ones as th...,"Within Tutankhamun’s tomb, two mummified foetu...",Ankhsenamun,pharaoh
215,"Raneferef, when you suddenly passed away, your...","I, Raneferef, had dreamed of a pyramid that wo...",The pyramid complex of Neferefre was known as ...,Raneferef,pharaoh
309,"Al Qurn, how did your distinctive pyramidal si...",I rise 450 meters above the western bank of th...,"El Qurn (Arabic: القرن, meaning ""The Horn"") is...",Al Qurn,landmark


In [ ]:
OUTPUT_DIR = r"C:\Uni\4th Year\GP\ECHO\data\chatbot\outputs\evaluation_data"
mid = len(shrunk_df) // 2
part1 = shrunk_df.iloc[:mid]
part2 = shrunk_df.iloc[mid:]

# 4. Save everything
shrunk_df.to_csv(os.path.join(OUTPUT_DIR, "shrunk_dataset_132.csv"), index=False)
part1.to_csv(os.path.join(OUTPUT_DIR, "eval_part_1.csv"), index=False)
part2.to_csv(os.path.join(OUTPUT_DIR, "eval_part_2.csv"), index=False)

In [16]:
# fix_all_contexts.py
import pandas as pd
import re

csv_path = r"C:\Uni\4th Year\GP\ECHO\data\chatbot\outputs\agent_responses\agent_responses_pt2.csv"
df = pd.read_csv(csv_path)

def clean_all_contexts(contexts_str):
    """
    Clean all malformed contexts:
    1. Remove contexts that start with punctuation only
    2. Remove empty contexts
    3. Strip whitespace
    4. Remove duplicates
    """
    if not isinstance(contexts_str, str):
        return "No context available"
    
    # Split by delimiter
    contexts = contexts_str.split('|||')
    
    cleaned = []
    for ctx in contexts:
        ctx = ctx.strip()
        
        # Skip empty
        if not ctx:
            continue
            
        # Skip if starts with only punctuation (., !, ?, etc.)
        if ctx and ctx[0] in '.!?,;:—':
            print(f"⚠️  Skipping fragment: {ctx[:50]}...")
            continue
        
        # Skip very short fragments (less than 20 chars)
        if len(ctx) < 20:
            print(f"⚠️  Skipping short fragment: {ctx}")
            continue
        
        cleaned.append(ctx)
    
    # If nothing left, return placeholder
    if not cleaned:
        return "No context available"
    
    # Remove duplicates while preserving order
    seen = set()
    unique_cleaned = []
    for ctx in cleaned:
        if ctx not in seen:
            seen.add(ctx)
            unique_cleaned.append(ctx)
    
    return '|||'.join(unique_cleaned)


# Apply cleaning to all rows
print(f"Cleaning contexts for {len(df)} samples...")

issues_found = 0
for idx, row in df.iterrows():
    old_contexts = row['contexts']
    new_contexts = clean_all_contexts(old_contexts)
    
    if old_contexts != new_contexts:
        issues_found += 1
        print(f"\n[Sample {idx}] Entity: {row['entity_name']}")
        print(f"  Old: {old_contexts[:100]}...")
        print(f"  New: {new_contexts[:100]}...")
        
        df.at[idx, 'contexts'] = new_contexts
        
        # Update context count
        new_count = len(new_contexts.split('|||'))
        df.at[idx, 'context_count'] = new_count

print(f"\n{'='*80}")
print(f"✅ Cleaned {issues_found} samples with context issues")
print(f"{'='*80}\n")

# Save fixed CSV
output_path = csv_path.replace(".csv", "_CLEANED.csv")
df.to_csv(output_path, index=False)

print(f"✅ Saved cleaned CSV to: {output_path}")
print(f"\nNext steps:")
print(f"1. Review the cleaned file")
print(f"2. Use this file for evaluation: {output_path}")

Cleaning contexts for 66 samples...
⚠️  Skipping fragment: . The enemy was defeated and the creation continue...

[Sample 0] Entity: Temple of Horus at Edfu
  Old: The earliest temple on the site was built by New Kingdom pharaohs Ramses I, Seti I and Ramses II, or...
  New: The earliest temple on the site was built by New Kingdom pharaohs Ramses I, Seti I and Ramses II, or...
⚠️  Skipping fragment: . In the myth, Set killed Osiris and cut his body ...

[Sample 1] Entity: Osiris (God)
  Old: . In the myth, Set killed Osiris and cut his body into pieces, which he scattered around Egypt; Isis...
  New: Osiris is the Egyptian Lord of the Underworld and Judge of the Dead, brother‑husband to Isis, and on...
⚠️  Skipping fragment: . Subsequent work included fitting new flooring in...

[Sample 2] Entity: Temple of Khonsu in Karnak
  Old: From 2006 to 2018 the American Research Center in Egypt performed conservation work overseen and car...
  New: From 2006 to 2018 the American Research Center 

In [14]:
# inspect_sample_35.py
import pandas as pd

# Load your agent responses CSV
df = pd.read_csv(r"C:\Uni\4th Year\GP\ECHO\data\chatbot\outputs\agent_responses\agent_responses_pt1.csv")

# Filter only successful ones (since evaluation only uses successful)
successful = df[df['success'] == True].reset_index(drop=True)

# Look at sample 35 (36th successful sample)
sample_35 = successful.iloc[34]

print("="*80)
print(f"SAMPLE 35 (Index 35)")
print("="*80)
print(f"\nEntity: {sample_35['entity_name']} ({sample_35['entity_type']})")
print(f"\nQuestion: {sample_35['question']}")
print(f"\nAnswer: {sample_35['answer'][:500]}...")
print(f"\nGround Truth: {sample_35['ground_truth'][:200]}...")
print(f"\nContexts: {sample_35['contexts'][:500]}...")
print(f"\nAnswer Length: {sample_35['answer_length']}")
print(f"\nContext Count: {sample_35['context_count']}")

SAMPLE 35 (Index 35)

Entity: Tomb of Petosiris (landmark)

Question: Tomb of Petosiris, how do the three stacked registers on your left wall orchestrate the drama of judgment and renewal, and why does the anonymity of the uppermost deity intensify the visitor’s experience of the sacred?

Answer: I have the three stacked registers carved upon my left wall, each a scene that I have watched countless visitors contemplate.

- **Lower register** – Here I stand before **Isis**, the mother‑goddess who shelters the dead and promises rebirth. Her presence opens the narrative with a protective embrace.
- **Middle register** – Above her I appear before **Osiris**, the ruler of the after‑life and the judge of souls. This placement moves the drama forward from protection to the moment of judgment, w...

Ground Truth: I, the Tomb of Petosiris, stage the soul’s journey in three ascending acts upon my left wall so that every visitor who walks my corridor must rise—literally and spiritually—through th